# 07 Calibration Evaluation

This notebook is the main numerical experiment results notebook.

It evaluates probabilistic VAR forecasts generated in `06_forecasts.ipynb` across:

- four controlled DGPs,
- four innovation models,
- multiple calibration and scoring metrics.

The central question is:

**When do flexible innovation models materially improve probabilistic forecast calibration under innovation misspecification?**

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.seeds import set_seed

from src.experiments.config import ExperimentConfig
from src.experiments.paths import result_dirs
from src.experiments.artifacts import (
    load_array_npz,
    save_table,
    save_json,
    save_current_figure,
)

from src.evaluation.metrics import (
    summarize_probabilistic_forecast,
    make_summary_row,
)

from src.calibration.reliability import reliability_table

from src.calibration.pit import (
    pit_values_multivariate_marginal,
    pit_histogram,
    pit_deviation_from_uniform,
)

from src.calibration.ece import calibration_curve_data

set_seed(123)

config = ExperimentConfig()

dgp_names = config.dgp_names

forecast_models = [
    "VAR",
    "RNN",
]

innovation_model_names = [
    "gaussian",
    "bootstrap",
    "student_t",
    "diffusion",
]

dirs = result_dirs(
    "07_calibration_evaluation",
    test=False,
)

forecast_dir = ROOT / "results" / "forecasts" / "06_forecasts"

interval = (0.05, 0.95)
nominal_levels = config.nominal_levels

forecast_store = {}

for forecast_model in forecast_models:
    forecast_store[forecast_model] = {}

    for dgp_name in dgp_names:
        forecast_store[forecast_model][dgp_name] = {}

        for innovation_model in innovation_model_names:
            path = (
                forecast_dir
                / forecast_model.lower()
                / dgp_name
                / f"{innovation_model}_forecast_paths.npz"
            )

            data = load_array_npz(path)

            forecast_store[forecast_model][dgp_name][innovation_model] = {
                "forecast_paths": data["forecast_paths"],
                "innovation_paths": data["innovation_paths"],
                "y_true": data["y_true"],
            }

            print(
                forecast_model,
                dgp_name,
                innovation_model,
                data["forecast_paths"].shape,
                data["y_true"].shape,
            )

In [ ]:
n_loaded = sum(
    len(forecast_store[forecast_model][dgp_name])
    for forecast_model in forecast_models
    for dgp_name in dgp_names
)

print("Loaded forecast objects:", n_loaded)

## Main Forecast Evaluation Table

The first evaluation step computes a unified set of probabilistic forecast metrics for each DGP and innovation model.

Metrics include:

- empirical interval coverage,
- average interval width,
- expected calibration error,
- PIT deviation from uniformity,
- CRPS,
- energy score,
- interval score.

Lower is better for ECE, PIT deviation, CRPS, energy score, and interval score. Coverage should be close to the nominal level.

In [ ]:
summary_rows = []
summary_objects = {}

for forecast_model in forecast_models:
    summary_objects[forecast_model] = {}

    for dgp_name in dgp_names:
        summary_objects[forecast_model][dgp_name] = {}

        for innovation_model in innovation_model_names:
            obj = forecast_store[forecast_model][dgp_name][innovation_model]

            summary = summarize_probabilistic_forecast(
                forecast_paths=obj["forecast_paths"],
                y_true=obj["y_true"],
                interval=interval,
                nominal_levels=nominal_levels,
            )

            summary_objects[forecast_model][dgp_name][innovation_model] = summary

            summary_rows.append(
                make_summary_row(
                    dgp_name=dgp_name,
                    forecast_model=forecast_model,
                    innovation_model=innovation_model,
                    summary=summary,
                )
            )

main_results_df = pd.DataFrame(summary_rows)

nominal_coverage = interval[1] - interval[0]

main_results_df["nominal_coverage"] = nominal_coverage
main_results_df["coverage_error"] = (
    main_results_df["avg_coverage"] - nominal_coverage
)
main_results_df["abs_coverage_error"] = (
    main_results_df["coverage_error"].abs()
)

main_results_df = main_results_df.sort_values(
    ["dgp", "forecast_model", "ece"]
)

save_table(
    main_results_df,
    dirs["tables"] / "main_calibration_results.csv",
)

main_results_df

In [ ]:
display_cols = [
    "forecast_model",
    "dgp",
    "innovation_model",
    "avg_coverage",
    "abs_coverage_error",
    "avg_width",
    "ece",
    "pit_deviation",
    "crps",
    "energy_score",
    "interval_score",
]

compact_results_df = main_results_df[
    display_cols
].copy()

compact_results_df

In [ ]:
lower_is_better_metrics = [
    "ece",
    "pit_deviation",
    "crps",
    "energy_score",
    "interval_score",
    "abs_coverage_error",
]

relative_rows = []

for forecast_model in forecast_models:
    for dgp_name in dgp_names:

        baseline = main_results_df[
            (main_results_df["forecast_model"] == forecast_model)
            & (main_results_df["dgp"] == dgp_name)
            & (main_results_df["innovation_model"] == "gaussian")
        ].iloc[0]

        for innovation_model in innovation_model_names:
            row = main_results_df[
                (main_results_df["forecast_model"] == forecast_model)
                & (main_results_df["dgp"] == dgp_name)
                & (main_results_df["innovation_model"] == innovation_model)
            ].iloc[0]

            out = {
                "forecast_model": forecast_model,
                "dgp": dgp_name,
                "innovation_model": innovation_model,
            }

            for metric in lower_is_better_metrics:
                baseline_value = baseline[metric]
                model_value = row[metric]

                if baseline_value == 0:
                    improvement = np.nan
                else:
                    improvement = (
                        baseline_value - model_value
                    ) / baseline_value

                out[f"{metric}_relative_improvement"] = improvement

            relative_rows.append(out)

relative_improvement_df = pd.DataFrame(relative_rows)

save_table(
    relative_improvement_df,
    dirs["tables"] / "relative_improvement_vs_gaussian.csv",
)

relative_improvement_df

In [ ]:
ranking_metrics = [
    "ece",
    "pit_deviation",
    "crps",
    "energy_score",
    "interval_score",
    "abs_coverage_error",
]

ranking_rows = []

for forecast_model in forecast_models:
    for dgp_name in dgp_names:

        dgp_df = main_results_df[
            (main_results_df["forecast_model"] == forecast_model)
            & (main_results_df["dgp"] == dgp_name)
        ]

        row = {
            "forecast_model": forecast_model,
            "dgp": dgp_name,
        }

        for metric in ranking_metrics:
            best_idx = dgp_df[metric].idxmin()
            best_row = dgp_df.loc[best_idx]

            row[f"best_{metric}_innovation"] = (
                best_row["innovation_model"]
            )

            row[f"best_{metric}_value"] = (
                best_row[metric]
            )

        ranking_rows.append(row)

headline_ranking_df = pd.DataFrame(
    ranking_rows
)

save_table(
    headline_ranking_df,
    dirs["tables"] / "headline_rankings.csv",
)

headline_ranking_df

In [ ]:
headline_ranking_df[
    [
        "forecast_model",
        "dgp",

        "best_ece_innovation",
        "best_ece_value",

        "best_crps_innovation",
        "best_crps_value",

        "best_energy_score_innovation",
        "best_energy_score_value",

        "best_interval_score_innovation",
        "best_interval_score_value",
    ]
]

In [ ]:
wins = []

metrics = [
    "ece",
    "crps",
    "energy_score",
    "interval_score",
]

for metric in metrics:

    counts = (
        headline_ranking_df[
            f"best_{metric}_innovation"
        ]
        .value_counts()
        .to_dict()
    )

    for innovation_model, n in counts.items():
        wins.append(
            {
                "metric": metric,
                "innovation_model": innovation_model,
                "wins": n,
            }
        )

win_df = pd.DataFrame(wins)

win_df.sort_values(
    ["metric", "wins"],
    ascending=[True, False],
)

In [ ]:
win_pivot = win_df.pivot(
    index="innovation_model",
    columns="metric",
    values="wins",
).fillna(0)

win_pivot

Diffusion innovation models provide the greatest benefit when
innovation distributions exhibit complex non-Gaussian structure
such as mixtures and heteroskedasticity.

The largest gains appear in forecast distribution quality
(energy score and interval score), while improvements in
calibration metrics such as ECE are more context dependent.

Bootstrap:
better calibration

Diffusion:
better probabilistic forecasts